# Data Prep from Silver to Gold

Requires silver parquet data

In gold layer we add features (especially spatial features like hexagon) and aggregate data from different datasets, etc.

In [1]:
import pandas as pd
import duckdb
import polars as pl
import numpy as np
import geopandas as gpd
from shapely import wkt
import h3
from shapely import from_wkb, from_wkt
from shapely.geometry.base import BaseGeometry
from pathlib import Path
import holidays


from datetime import datetime
import math

from run_config import (
    PATHS,
    RUN_MODE,
    PROCESSED_DIR,
    H3_RESOLUTION,
    DEFAULT_H3_RESOLUTION,
    GOLD_TIME_UNITS,
    START_DATE,
    END_DATE,
)

BRONZE_CENSUS_TRACTS = PATHS.bronze_census_tracts
BRONZE_COMMUNITY_AREA = PATHS.bronze_community_areas

SILVER_CENSUS_TRACTS = PATHS.silver_census_tracts
SILVER_COMMUNITY_AREA = PATHS.silver_community_areas
BRONZE_POIS = PATHS.bronze_osm_geo

SILVER_TAXI_COMMUNITY_UNFILTERED_PATH = PATHS.silver_taxi_community_areas_unfiltered
SILVER_TAXI_COMMUNITY_PATH = PATHS.silver_taxi_community_areas
SILVER_TAXI_CENSUS_PATH = PATHS.silver_taxi_census_tracts
SILVER_TAXI_HEXAGON_PATHS = PATHS.silver_taxi_hexagons
SILVER_TAXI_HEXAGON_PATH = SILVER_TAXI_HEXAGON_PATHS[DEFAULT_H3_RESOLUTION]
SILVER_WEATHER_PATH = PATHS.silver_weatherdata
SILVER_HEXAGON_PATHS = PATHS.silver_hexagons
SILVER_HEXAGON_PATH = SILVER_HEXAGON_PATHS[DEFAULT_H3_RESOLUTION]

GOLD_TAXI_PATH = PATHS.gold_taxi_trips
GOLD_WEATHER_PATH = PATHS.gold_weatherdata
GOLD_DEMAND_PATHS = {
    "1h": (
        PATHS.gold_1h_demand_hexagons,
        PATHS.gold_1h_demand_census_tracts,
        PATHS.gold_1h_demand_community_areas,
        PATHS.gold_1h_demand_community_area_unfiltered,
    ),
    "24h": (
        PATHS.gold_24h_demand_hexagons,
        PATHS.gold_24h_demand_census_tracts,
        PATHS.gold_24h_demand_community_areas,
        PATHS.gold_24h_demand_community_area_unfiltered,
    ),
    "4h": (
        PATHS.gold_4h_demand_hexagons,
        PATHS.gold_4h_demand_census_tracts,
        PATHS.gold_4h_demand_community_areas,
        PATHS.gold_4h_demand_community_area_unfiltered,
    ),
}
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

configured_start = datetime.fromisoformat(START_DATE)
configured_end = datetime.fromisoformat(END_DATE)

def scan_taxi_in_configured_period(path: Path) -> pl.LazyFrame:
    """Read taxi rows in the loader's half-open interval [start, end)."""
    return pl.scan_parquet(path).filter(
        (pl.col("trip_start_timestamp") >= configured_start)
        & (pl.col("trip_start_timestamp") < configured_end)
    )

# Effective processing period: intersection of configured and available taxi data
taxi_min_ts, taxi_max_ts = (
    scan_taxi_in_configured_period(SILVER_TAXI_COMMUNITY_UNFILTERED_PATH)
    .select(
        pl.col("trip_start_timestamp").min().alias("min_ts"),
        pl.col("trip_start_timestamp").max().alias("max_ts"),
    )
    .collect()
    .row(0)
)

if taxi_min_ts is None or taxi_max_ts is None:
    raise ValueError(f"No taxi timestamps found in {SILVER_TAXI_COMMUNITY_UNFILTERED_PATH}")

bounded_start = max(configured_start, taxi_min_ts)
bounded_end = min(configured_end, taxi_max_ts)

if bounded_start > bounded_end:
    raise ValueError(
        "Configured period and available taxi data do not overlap: "
        f"configured={START_DATE}..{END_DATE}, "
        f"available={taxi_min_ts.isoformat()}..{taxi_max_ts.isoformat()}"
    )

START_TS = bounded_start.isoformat()
END_TS = bounded_end.isoformat()

print(f"Run mode: {RUN_MODE}")
print(f"Gold time units: {GOLD_TIME_UNITS}")
print(f"Configured period: {START_DATE} to {END_DATE}")
print(f"Available taxi period: {taxi_min_ts} to {taxi_max_ts}")
print(f"Effective raw processing period: {START_TS} to {END_TS}")
for time_unit in GOLD_TIME_UNITS:
    print(f"Gold {time_unit} outputs: {GOLD_DEMAND_PATHS[time_unit]}")

Run mode: sample
Gold time units: ('1h', '4h', '24h')
Configured period: 2024-01-01T00:00:00 to 2026-05-01T00:00:00
Available taxi period: 2026-04-13 00:00:00 to 2026-04-26 23:45:00
Effective raw processing period: 2026-04-13T00:00:00 to 2026-04-26T23:45:00
Gold 1h outputs: ({7: PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/GOLD_1H_DEMAND_HEXAGON_7.parquet'), 8: PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/GOLD_1H_DEMAND_HEXAGON_8.parquet')}, PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/GOLD_1H_DEMAND_CENSUS_TRACTS.parquet'), PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet'), PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/GOLD_1H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet'))
Gold 4h outputs: ({7: PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/GOLD_4H_DEMAND_HEXA

Nachdem in silver alle Duplikate bereinigt wurden, können nun trip_id und taxi_id gedropped werden

## Taxi Data

### Adding hexagon

First we add hexagon data and then compare the hexagons of trips against the city boundaries of chicago. We only want trips with pickup hexagon in the city boundaries

In [2]:
def h3_from_row(row, lat_col: str, lon_col: str, resolution: int):
    """Return an H3 cell or None when either centroid coordinate is missing."""
    lat, lon = row[lat_col], row[lon_col]
    if lat is None or lon is None:
        return None
    return h3.latlng_to_cell(lat, lon, resolution)


def h3_expr(lat_col: str, lon_col: str, resolution: int, alias: str) -> pl.Expr:
    return (
        pl.struct([lat_col, lon_col])
        .map_elements(
            lambda row: h3_from_row(row, lat_col, lon_col, resolution),
            return_dtype=pl.String,
        )
        .alias(alias)
    )


taxi_with_h3 = (
    scan_taxi_in_configured_period(SILVER_TAXI_HEXAGON_PATH)
    .with_columns([
        h3_expr("pickup_centroid_latitude", "pickup_centroid_longitude", DEFAULT_H3_RESOLUTION, "pickup_h3_cell"),
        # dropoff coordinates can be null (~8%), h3_from_row yields None for those
        h3_expr("dropoff_centroid_latitude", "dropoff_centroid_longitude", DEFAULT_H3_RESOLUTION, "dropoff_h3_cell"),
    ])
)

# Check if there are trips without hexagon in Chicagos boundaries
valid_chicago_h3_cells = (
    pl.scan_parquet(SILVER_HEXAGON_PATH)
    .select("h3_cell")
)

count_before = (
    taxi_with_h3
    .select(pl.len().alias("n_rows_before"))
    .collect()
)

taxi_with_h3_chicago_only = (
    taxi_with_h3
    .join(
        valid_chicago_h3_cells,
        left_on="pickup_h3_cell",
        right_on="h3_cell",
        how="inner",
    )
)

count_after = (
    taxi_with_h3_chicago_only
    .select(pl.len().alias("n_rows_after"))
    .collect()
)

print(count_before)
print(count_after)

dropoff_h3_quality = (
    taxi_with_h3_chicago_only
    .select([
        pl.len().alias("trips"),
        pl.col("dropoff_h3_cell").is_null().sum().alias("missing_dropoff_h3"),
    ])
    .with_columns(
        (100 * pl.col("missing_dropoff_h3") / pl.col("trips")).alias("missing_dropoff_h3_pct")
    )
    .collect()
)
print(dropoff_h3_quality)

/var/folders/sw/7cvjbzt5623085bdv7hpg6vw0000gn/T/ipykernel_41523/829413567.py:54: UserWarning: Extension type 'geoarrow.wkb' is not registered; loading as its storage type.

To avoid this warning, register the extension type or set environment variable 'POLARS_UNKNOWN_EXTENSION_TYPE_BEHAVIOR' to 'load_as_storage' or 'load_as_extension'.

In Polars 2.0, the default behavior will change to 'load_as_extension'.
  .collect()


shape: (1, 1)
┌───────────────┐
│ n_rows_before │
│ ---           │
│ u32           │
╞═══════════════╡
│ 130206        │
└───────────────┘
shape: (1, 1)
┌──────────────┐
│ n_rows_after │
│ ---          │
│ u32          │
╞══════════════╡
│ 130206       │
└──────────────┘
shape: (1, 3)
┌────────┬────────────────────┬────────────────────────┐
│ trips  ┆ missing_dropoff_h3 ┆ missing_dropoff_h3_pct │
│ ---    ┆ ---                ┆ ---                    │
│ u32    ┆ u32                ┆ f64                    │
╞════════╪════════════════════╪════════════════════════╡
│ 130206 ┆ 3555               ┆ 2.730289               │
└────────┴────────────────────┴────────────────────────┘


### Save gold version


In [3]:
taxi_with_h3_chicago_only.sink_parquet(GOLD_TAXI_PATH)

print(f"Gold H3-filtered taxi parquet written to: {GOLD_TAXI_PATH}")

Gold H3-filtered taxi parquet written to: /Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/gold_taxi_sample.parquet


## Weather Data

Create hourly indexed dataset for weather data from 01.01.2024 to 24.05.2026

In [4]:
WEATHER_FLAG_COLUMNS = [
    "weather_rain", "weather_snow", "weather_fog_mist",
    "weather_thunder", "weather_freezing", "precipitation_trace",
    "weather_qc_corrected",
]
EXPECTED_WEATHER_STATIONS = ("MDW", "ORD", "IGQ")
SKY_SEVERITY = {"CLR": 0, "FEW": 1, "SCT": 2, "BKN": 3, "OVC": 4, "VV": 5}


def create_hourly_weather_gold(
    df: pd.DataFrame,
    time_unit: str,
    start_ts: str = START_TS,
    end_ts: str = END_TS,
) -> pd.DataFrame:
    """
    Create weather features aggregated to time_unit.

    Build one complete weather series per airport. Spatial demand regions are
    joined to their nearest station later; observations are never averaged
    across MDW, ORD, and IGQ.
    """

    df = df.copy()
    df["station"] = df["station"].astype("string").str.strip().str.upper()

    # The IEM download is requested with tz=America/Chicago. Taxi and weather
    # timestamps therefore both represent local Chicago wall-clock time.
    df["valid"] = pd.to_datetime(df["valid"], errors="coerce")
    df = df.dropna(subset=["valid"])

    # Normalize timezone-aware inputs to local, timezone-naive timestamps.
    if df["valid"].dt.tz is not None:
        df["valid"] = (
            df["valid"].dt.tz_convert("America/Chicago").dt.tz_localize(None)
        )

    start_ts = pd.Timestamp(start_ts)
    end_ts = pd.Timestamp(end_ts)
    if start_ts.tzinfo is not None:
        start_ts = start_ts.tz_convert("America/Chicago").tz_localize(None)
    if end_ts.tzinfo is not None:
        end_ts = end_ts.tz_convert("America/Chicago").tz_localize(None)

    # Keep the same half-open interval used by the loader: [start, end).
    df = df[(df["valid"] >= start_ts) & (df["valid"] < end_ts)]
    if df.empty:
        raise ValueError(
            f"No weather observations overlap the processing period {start_ts}..{end_ts}"
        )
    missing_stations = set(EXPECTED_WEATHER_STATIONS) - set(df["station"].dropna())
    if missing_stations:
        raise ValueError(f"Missing weather stations: {sorted(missing_stations)}")

    required_columns = {
        "station", "tmpc", "relh", "sknt", "p01m", "vsby",
        "skyc1_severity", "wind_dir_sin", "wind_dir_cos",
        *WEATHER_FLAG_COLUMNS,
    }
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(
            "Silver weather is missing normalized columns: "
            f"{sorted(missing_columns)}. Rerun Silver first."
        )

    numeric_columns = [
        "tmpc", "relh", "sknt", "p01m", "vsby", "skyc1_severity",
        "wind_dir_sin", "wind_dir_cos", *WEATHER_FLAG_COLUMNS,
    ]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    # Defensive station-hour reduction. Silver normally already guarantees
    # this, but keeping it here prevents accidental precipitation overcounting.
    df["valid_hour"] = df["valid"].dt.floor("h")
    quality_columns = [
        "tmpc", "relh", "sknt", "p01m", "vsby", "skyc1_severity",
    ]
    df["_quality"] = df[quality_columns].notna().sum(axis=1)
    station_hour = (
        df.sort_values(["station", "valid_hour", "_quality", "valid"])
        .drop_duplicates(["station", "valid_hour"], keep="last")
        .drop(columns="_quality")
    )

    full_hourly_index = pd.date_range(
        start=start_ts.floor("h"),
        end=end_ts,
        freq="1h",
        inclusive="left",
        name="valid_hour",
    )

    station_value_columns = [
        "tmpc", "relh", "sknt", "p01m", "vsby", "skyc1_severity",
        "wind_dir_sin", "wind_dir_cos", *WEATHER_FLAG_COLUMNS,
    ]
    core_columns = ["tmpc", "relh", "sknt", "vsby"]
    station_frames = []

    for station in EXPECTED_WEATHER_STATIONS:
        hourly = (
            station_hour.loc[station_hour["station"].eq(station)]
            .set_index("valid_hour")[station_value_columns]
            .sort_index()
        )
        observed_hours = hourly.index.copy()
        hourly = hourly.reindex(full_hourly_index)

        # Retain missingness before producing the model-compatible complete grid.
        hourly["station_observed"] = hourly.index.isin(observed_hours).astype("int8")
        hourly["weather_imputed"] = hourly[core_columns].isna().any(axis=1).astype("int8")
        hourly["precipitation_missing"] = hourly["p01m"].isna().astype("int8")
        hourly[core_columns] = hourly[core_columns].interpolate(
            method="time", limit_direction="both"
        )
        hourly[["wind_dir_sin", "wind_dir_cos"]] = hourly[
            ["wind_dir_sin", "wind_dir_cos"]
        ].interpolate(method="time", limit_direction="both").fillna(0.0)
        hourly["p01m"] = hourly["p01m"].fillna(0.0)
        hourly["skyc1_severity"] = hourly["skyc1_severity"].ffill().bfill()
        hourly[WEATHER_FLAG_COLUMNS] = hourly[WEATHER_FLAG_COLUMNS].fillna(0)

        if hourly[core_columns].isna().any().any():
            raise ValueError(f"Weather interpolation left missing values for {station}")

        # Wider intervals are aggregated within a station, never across stations.
        if time_unit != "1h":
            period_aggregations = {
                "tmpc": "mean",
                "relh": "mean",
                "sknt": "mean",
                "p01m": "sum",
                "vsby": "min",
                "skyc1_severity": "max",
                "wind_dir_sin": "mean",
                "wind_dir_cos": "mean",
                "station_observed": "min",
                "weather_imputed": "max",
                "precipitation_missing": "max",
                **{column: "max" for column in WEATHER_FLAG_COLUMNS},
            }
            hourly = hourly.groupby(pd.Grouper(freq=time_unit)).agg(period_aggregations)

        hourly["station"] = station
        station_frames.append(hourly.reset_index())

    gold = pd.concat(station_frames, ignore_index=True)

    # Emit a fixed, whitespace-free cloud schema for every station and period.
    for sky_label, severity in SKY_SEVERITY.items():
        gold[f"skyc1_{sky_label}"] = gold["skyc1_severity"].eq(severity).astype("int8")
    gold = gold.drop(columns="skyc1_severity")

    integer_columns = [
        "station_observed", "weather_imputed", "precipitation_missing",
        *WEATHER_FLAG_COLUMNS,
    ]
    gold[integer_columns] = gold[integer_columns].astype("int8")
    gold = gold.sort_values(["valid_hour", "station"]).reset_index(drop=True)
    expected_rows_per_hour = gold.groupby("valid_hour")["station"].nunique()
    if not expected_rows_per_hour.eq(len(EXPECTED_WEATHER_STATIONS)).all():
        raise ValueError("Gold weather does not contain all stations in every period")
    return gold

weather_silver = pd.read_parquet(SILVER_WEATHER_PATH)
print(f"Silver weather rows loaded once: {len(weather_silver):,}")
WEATHER_STATION_LOCATIONS = (
    weather_silver.loc[
        weather_silver["station"].isin(EXPECTED_WEATHER_STATIONS),
        ["station", "lat", "lon"],
    ]
    .groupby("station", as_index=False)[["lat", "lon"]]
    .median()
    .sort_values("station")
    .reset_index(drop=True)
)
if set(WEATHER_STATION_LOCATIONS["station"]) != set(EXPECTED_WEATHER_STATIONS):
    raise ValueError("Could not derive coordinates for all weather stations")

# Persist the standalone 1h weather feature table used by visual analyses.
weather_gold_start = pd.Timestamp(START_TS).floor("1h")
weather_gold_end = min(
    pd.Timestamp(END_DATE),
    pd.Timestamp(END_TS).floor("1h") + pd.Timedelta(hours=1),
)
weather_gold_1h = create_hourly_weather_gold(
    weather_silver,
    time_unit="1h",
    start_ts=weather_gold_start.isoformat(),
    end_ts=weather_gold_end.isoformat(),
)
weather_gold_1h.to_parquet(GOLD_WEATHER_PATH, index=False)
print(f"Written 1h Gold weather: {GOLD_WEATHER_PATH} ({len(weather_gold_1h):,} rows)")

Silver weather rows loaded once: 1,005
Written 1h Gold weather: /Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/gold_weather.parquet (1,008 rows)


## Points of Interest

In [5]:
pois = gpd.read_parquet(BRONZE_POIS)

print("\nPOIs:")
print(pois.shape)
print(pois.columns)
print(pois.crs)
print(type(pois.geometry.iloc[0]))



POIs:
(9117, 21)
Index(['poi_category', 'name', 'amenity', 'shop', 'railway', 'tourism',
       'historic', 'man_made', 'brand', 'operator', 'opening_hours',
       'addr_housenumber', 'addr_street', 'addr_city', 'website', 'phone',
       'geom_type_original', 'area_m2', 'lat', 'lon', 'geometry'],
      dtype='object')
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accura

### Hexagon

In [6]:
hexagons = gpd.read_parquet(SILVER_HEXAGON_PATH)

print("Hexagons:")
print(hexagons.shape)
print(hexagons.columns)
print(hexagons.crs)
print(type(hexagons.geometry.iloc[0]))

# CRS prüfen
print("Hexagon CRS:", hexagons.crs)
print("POI CRS:", pois.crs)

# Falls CRS unterschiedlich sind: POIs auf CRS der Hexagons bringen
if pois.crs != hexagons.crs:
    pois = pois.to_crs(hexagons.crs)

print("CRS identisch:", pois.crs == hexagons.crs)

pois_with_h3 = gpd.sjoin(
    pois,
    hexagons[["h3_cell", "geometry"]],
    how="left",
    predicate="within"
)

pois_with_h3.head()
# technische Join-Spalte entfernen
pois_with_h3 = pois_with_h3.drop(columns=["index_right"])

# prüfen, wie viele POIs keine h3_cell bekommen haben
missing_h3 = pois_with_h3["h3_cell"].isna().sum()

print("POIs gesamt:", len(pois_with_h3))
print("POIs ohne h3_cell:", missing_h3)
print("Anteil ohne h3_cell:", round(missing_h3 / len(pois_with_h3) * 100, 2), "%")

pois_with_h3[["name", "poi_category", "lat", "lon", "h3_cell"]].head()

print("POIs vorher:", len(pois))
print("POIs nach Join:", len(pois_with_h3))

duplicate_rows = len(pois_with_h3) - len(pois)

print("Zusätzliche Zeilen durch Join:", duplicate_rows)

Hexagons:
(163, 3)
Index(['h3_cell', 'h3_resolution', 'geometry'], dtype='object')
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "a

### Census Tract

In [7]:
census_tracts = gpd.read_parquet(SILVER_CENSUS_TRACTS)

print("Census Tracts:")
print(census_tracts.shape)
print(census_tracts.columns)
print(census_tracts.crs)
print(type(census_tracts.geometry.iloc[0]))

# CRS prüfen
print("Census Tract CRS:", census_tracts.crs)
print("POI CRS:", pois.crs)

# Falls CRS unterschiedlich sind: POIs auf CRS der Hexagons bringen
if pois.crs != census_tracts.crs:
    pois = pois.to_crs(census_tracts.crs)

print("CRS identisch:", pois.crs == census_tracts.crs)

pois_with_census_tract = gpd.sjoin(
    pois_with_h3,
    census_tracts[["CENSUS_T_1", "geometry"]],
    how="left",
    predicate="within"
)

pois_with_census_tract.head()
# technische Join-Spalte entfernen
pois_with_census_tract = pois_with_census_tract.drop(columns=["index_right"])

# prüfen, wie viele POIs keine h3_cell bekommen haben
missing_census_tract = pois_with_census_tract["CENSUS_T_1"].isna().sum()

print("POIs gesamt:", len(pois_with_census_tract))
print("POIs ohne census tract:", missing_census_tract)
print("Anteil ohne census tract", round(missing_census_tract / len(pois_with_census_tract) * 100, 2), "%")

pois_with_census_tract[["name", "poi_category", "lat", "lon", "CENSUS_T_1"]].head()

print("POIs vorher:", len(pois))
print("POIs nach Join:", len(pois_with_census_tract))

duplicate_rows = len(pois_with_census_tract) - len(pois)

print("Zusätzliche Zeilen durch Join:", duplicate_rows)

Census Tracts:
(878, 18)
Index(['OBJECTID', 'CENSUS_TRA', 'CENSUS_T_1', 'TRACT_FIPS', 'TRACT_CENT',
       'TRACT_CE_1', 'TRACT_CE_2', 'TRACT_CE_3', 'TRACT_COMM', 'TRACT_NUMA',
       'TRACT_CENS', 'PERIMETER', 'DATA_ADMIN', 'TRACT_CREA', 'TRACT_CR_1',
       'SHAPE_AREA', 'SHAPE_LEN', 'geometry'],
      dtype='object')
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accurac

### Community Area

In [8]:
community_area = gpd.read_parquet(SILVER_COMMUNITY_AREA)

print("Community Area:")
print(community_area.shape)
print(community_area.columns)
print(community_area.crs)
print(type(community_area.geometry.iloc[0]))

# CRS prüfen
print("Community Area CRS:", community_area.crs)
print("POI CRS:", pois.crs)

# Falls CRS unterschiedlich sind: POIs auf CRS der Hexagons bringen
if pois.crs != community_area.crs:
    pois = pois.to_crs(community_area.crs)

print("CRS identisch:", pois.crs == community_area.crs)

pois_with_community_area = gpd.sjoin(
    pois_with_census_tract,
    community_area[["AREA_NUM_1", "geometry"]],
    how="left",
    predicate="within"
)

pois_with_community_area.head()
# technische Join-Spalte entfernen
pois_with_community_area = pois_with_community_area.drop(columns=["index_right"])

# prüfen, wie viele POIs keine h3_cell bekommen haben
missing_community_area = pois_with_community_area["AREA_NUM_1"].isna().sum()

print("POIs gesamt:", len(pois_with_community_area))
print("POIs ohne community area:", missing_community_area)
print("Anteil ohne community area", round(missing_community_area / len(pois_with_community_area) * 100, 2), "%")

pois_with_community_area[["name", "poi_category", "lat", "lon", "AREA_NUM_1"]].head()

print("POIs vorher:", len(pois))
print("POIs nach Join:", len(pois_with_community_area))

duplicate_rows = len(pois_with_community_area) - len(pois)

print("Zusätzliche Zeilen durch Join:", duplicate_rows)

Community Area:
(77, 6)
Index(['AREA_NUMBE', 'COMMUNITY', 'AREA_NUM_1', 'SHAPE_AREA', 'SHAPE_LEN',
       'geometry'],
      dtype='object')
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "no

In [9]:
# Drop POIs only if all spatial assignments are missing
pois_with_spatials = pois_with_community_area.dropna(
    subset=["h3_cell", "CENSUS_T_1", "AREA_NUM_1"],
    how="all"
)

POI_COLUMNS = ["food_drink", "landmark", "shop", "train_station"]

def group_pois_by_spatial(spatial_col):
    
    poi_h3_categories = pois_with_spatials[[spatial_col, "poi_category"]].copy()

    poi_h3_categories.head()
    
    poi_counts_long = (
    poi_h3_categories
        .groupby([spatial_col, "poi_category"])
        .size()
        .reset_index(name="poi_count")
    )

    poi_counts_long.head()

    poi_counts_wide = (
        poi_counts_long
        .pivot_table(
            index=spatial_col,
            columns="poi_category",
            values="poi_count",
            fill_value=0
        )
        .reset_index()
    )

    poi_counts_wide.head()
    
    poi_counts_wide.columns.name = None

    poi_counts_wide.head()
    
    return poi_counts_wide

pois_grouped_hexagon = group_pois_by_spatial("h3_cell")
pois_grouped_census_tract = group_pois_by_spatial("CENSUS_T_1")
pois_grouped_community_area = group_pois_by_spatial("AREA_NUM_1")

## Hourly Dataset

### Preparations

Here we create an hourly dataset, whereby each hour contains a row for each hexagon. 

We use cyclic encoding for columns like hour, weekday or month to be aware of the distance between time instances (e. g. hour 23 -> 0).

And we merge with the weather data.

These steps are done before the cross join as they are only dependend on time and not on the spatial unit

In [10]:
START = datetime.fromisoformat(START_TS).date()
END = datetime.fromisoformat(END_TS).date()

il_holidays = holidays.country_holidays(
    country="US",
    subdiv="IL", # Illinois = relevant for Chicago
    years=[2024, 2025, 2026]
)

holiday_df = pl.DataFrame(
    [
        {
            "date": d,
            "is_holiday": 1,
        }
        for d, name in sorted(il_holidays.items())
        if START <= d <= END
    ],
    schema={"date": pl.Date, "is_holiday": pl.Int8},
)

In [11]:
def effective_weather_bounds(time_unit: str) -> tuple[pd.Timestamp, pd.Timestamp]:
    """Return the shared half-open time range for temporal and weather data."""
    effective_start = pd.Timestamp(bounded_start).floor(time_unit)
    last_taxi_bucket = pd.Timestamp(bounded_end).floor(time_unit)
    effective_end_exclusive = min(
        pd.Timestamp(configured_end),
        last_taxi_bucket + pd.tseries.frequencies.to_offset(time_unit),
    )
    return effective_start, effective_end_exclusive


def nearest_weather_station_assignment(
    points: pd.DataFrame,
    id_column: str,
    latitude_column: str = "latitude",
    longitude_column: str = "longitude",
) -> pd.DataFrame:
    """Assign every spatial unit to its nearest airport using haversine distance."""
    required = {id_column, latitude_column, longitude_column}
    missing = required - set(points.columns)
    if missing:
        raise ValueError(f"Station assignment is missing columns: {sorted(missing)}")
    if points[list(required)].isna().any().any():
        raise ValueError("Station assignment points contain null identifiers or coordinates")

    point_lat = np.radians(points[latitude_column].to_numpy(dtype=float))[:, None]
    point_lon = np.radians(points[longitude_column].to_numpy(dtype=float))[:, None]
    station_lat = np.radians(
        WEATHER_STATION_LOCATIONS["lat"].to_numpy(dtype=float)
    )[None, :]
    station_lon = np.radians(
        WEATHER_STATION_LOCATIONS["lon"].to_numpy(dtype=float)
    )[None, :]
    delta_lat = station_lat - point_lat
    delta_lon = station_lon - point_lon
    haversine_a = (
        np.sin(delta_lat / 2.0) ** 2
        + np.cos(point_lat) * np.cos(station_lat) * np.sin(delta_lon / 2.0) ** 2
    )
    distances_km = 6371.0088 * 2.0 * np.arctan2(
        np.sqrt(haversine_a), np.sqrt(1.0 - haversine_a)
    )
    nearest_index = distances_km.argmin(axis=1)

    assignment = points[[id_column]].copy()
    assignment["weather_station"] = (
        WEATHER_STATION_LOCATIONS["station"].to_numpy()[nearest_index]
    )
    assignment["weather_station_distance_km"] = distances_km[
        np.arange(len(points)), nearest_index
    ]
    if assignment[id_column].duplicated().any():
        raise ValueError(f"Duplicate spatial identifiers in {id_column} station assignment")
    return assignment


def create_temporal_features(time_unit: str) -> pl.LazyFrame:
    """Create time and holiday features without applying city-wide weather."""
    effective_start, effective_end_exclusive = effective_weather_bounds(time_unit)
    hourly_index = pd.date_range(
        start=effective_start,
        end=effective_end_exclusive,
        freq=time_unit,
        inclusive="left",
        name="datetime_hour",
    )
    hours = (
        pl.from_pandas(pd.DataFrame({"datetime_hour": hourly_index}))
        .with_columns(
            pl.col("datetime_hour")
            .cast(pl.Datetime("us"))
            .dt.truncate(time_unit)
        )
    )

    hours_with_features = (
        hours
        .with_columns([
            pl.col("datetime_hour").dt.month().alias("month"),
            pl.col("datetime_hour").dt.weekday().alias("weekday"),
            pl.col("datetime_hour").dt.hour().alias("hour"),
        ])
        .with_columns([
            ((2 * math.pi * (pl.col("month") - 1) / 12).sin()).alias("month_sin"),
            ((2 * math.pi * (pl.col("month") - 1) / 12).cos()).alias("month_cos"),
            ((2 * math.pi * (pl.col("weekday") - 1) / 7).sin()).alias("weekday_sin"),
            ((2 * math.pi * (pl.col("weekday") - 1) / 7).cos()).alias("weekday_cos"),
            ((2 * math.pi * pl.col("hour") / 24).sin()).alias("hour_sin"),
            ((2 * math.pi * pl.col("hour") / 24).cos()).alias("hour_cos"),
        ])
    )

    return (
        hours_with_features.lazy()
        .with_columns(pl.col("datetime_hour").dt.date().alias("date"))
        .join(holiday_df.lazy(), on="date", how="left")
        .with_columns(pl.col("is_holiday").fill_null(0).cast(pl.Int8))
    )


def create_weather_features(time_unit: str) -> pl.LazyFrame:
    """Create a complete station-specific weather table for one interval."""
    effective_start, effective_end_exclusive = effective_weather_bounds(time_unit)
    weather_gold = create_hourly_weather_gold(
        weather_silver,
        time_unit=time_unit,
        start_ts=effective_start.isoformat(),
        end_ts=effective_end_exclusive.isoformat(),
    )
    if weather_gold["tmpc"].isna().any():
        raise ValueError(f"Weather contains missing temperatures for {time_unit}")
    return (
        pl.from_pandas(weather_gold)
        .with_columns(
            pl.col("valid_hour").cast(pl.Datetime("us")).alias("datetime_hour")
        )
        .drop("valid_hour")
        .rename({"station": "weather_station"})
        .lazy()
    )


def attach_nearest_station_weather(
    temporal_features: pl.LazyFrame,
    spatial_context: pl.LazyFrame,
    weather_features: pl.LazyFrame,
) -> pl.LazyFrame:
    """Cross time with space, then join weather from the assigned station."""
    return (
        temporal_features
        .join(spatial_context, how="cross")
        .join(
            weather_features,
            on=["datetime_hour", "weather_station"],
            how="left",
        )
        .with_columns([
            (pl.col("weather_station") == station)
            .cast(pl.Int8)
            .alias(f"weather_station_{station}")
            for station in EXPECTED_WEATHER_STATIONS
        ])
        .drop("weather_station")
    )

### Hexagon Hourly

In [12]:
def create_hexagon_spatial_context(resolution: int) -> pl.LazyFrame:
    hexagon_cells = (
        pl.scan_parquet(SILVER_HEXAGON_PATHS[resolution])
        .select(pl.col("h3_cell"))
        .unique()
        .collect()
        .to_pandas()
    )
    coordinates = [h3.cell_to_latlng(cell) for cell in hexagon_cells["h3_cell"]]
    hexagon_cells["latitude"] = [latitude for latitude, _ in coordinates]
    hexagon_cells["longitude"] = [longitude for _, longitude in coordinates]
    station_assignment = nearest_weather_station_assignment(
        hexagon_cells, id_column="h3_cell"
    )

    poi_frame = pois[["lat", "lon", "poi_category"]].dropna(
        subset=["lat", "lon"]
    ).copy()
    poi_frame["h3_cell"] = [
        h3.latlng_to_cell(lat, lon, resolution)
        for lat, lon in zip(poi_frame["lat"], poi_frame["lon"])
    ]
    poi_counts = (
        poi_frame.groupby(["h3_cell", "poi_category"])
        .size().unstack(fill_value=0).reset_index()
    )
    for column in POI_COLUMNS:
        if column not in poi_counts:
            poi_counts[column] = 0

    print(f"H3 resolution {resolution} weather-station assignment:")
    print(station_assignment["weather_station"].value_counts().sort_index())
    return (
        pl.from_pandas(station_assignment).lazy()
        .join(
            pl.from_pandas(poi_counts[["h3_cell", *POI_COLUMNS]]).lazy(),
            on="h3_cell", how="left",
        )
        .with_columns([pl.col(col).fill_null(0) for col in POI_COLUMNS])
        .with_columns(pl.lit(resolution).cast(pl.Int8).alias("h3_resolution"))
    )


hexagon_contexts = {
    resolution: create_hexagon_spatial_context(resolution)
    for resolution in H3_RESOLUTION
}

H3 resolution 7 weather-station assignment:
weather_station
IGQ     14
MDW    103
ORD     46
Name: count, dtype: int64
H3 resolution 8 weather-station assignment:
weather_station
IGQ     75
MDW    647
ORD    254
Name: count, dtype: int64


### Census Tract Hourly

In [13]:
census_tract_points = (
    pl.scan_parquet(SILVER_CENSUS_TRACTS)
    .select([
        pl.col("CENSUS_T_1").cast(pl.String).alias("census_tract"),
        pl.col("TRACT_CE_3").cast(pl.Float64).alias("latitude"),
        pl.col("TRACT_CE_2").cast(pl.Float64).alias("longitude"),
    ])
    .unique(subset=["census_tract"])
    .collect()
    .to_pandas()
)
census_station_assignment = nearest_weather_station_assignment(
    census_tract_points,
    id_column="census_tract",
)
census_tracts = pl.from_pandas(census_station_assignment).lazy()
print("Census weather-station assignment:")
print(census_station_assignment["weather_station"].value_counts().sort_index())

census_tracts_with_pois = (
    census_tracts
    .join(
        pl.from_pandas(pois_grouped_census_tract).lazy(),
        left_on="census_tract",
        right_on="CENSUS_T_1",
        how="left"
    )
    .with_columns([pl.col(col).fill_null(0) for col in POI_COLUMNS])
)

Census weather-station assignment:
weather_station
IGQ     10
MDW    736
ORD    132
Name: count, dtype: int64


### Community Area Hourly

In [14]:
community_geometries = gpd.read_parquet(SILVER_COMMUNITY_AREA)
community_projected = community_geometries.to_crs(26916)
community_points_projected = community_projected.geometry.representative_point()
community_points_wgs84 = gpd.GeoSeries(
    community_points_projected,
    crs=community_projected.crs,
).to_crs(4326)
community_area_points = pd.DataFrame({
    "community_area": community_geometries["AREA_NUM_1"].astype("string"),
    "latitude": community_points_wgs84.y,
    "longitude": community_points_wgs84.x,
})
community_station_assignment = nearest_weather_station_assignment(
    community_area_points,
    id_column="community_area",
)
community_area = pl.from_pandas(community_station_assignment).lazy()
print("Community-area weather-station assignment:")
print(community_station_assignment["weather_station"].value_counts().sort_index())

community_area_with_pois = (
    community_area
    .join(
        pl.from_pandas(pois_grouped_community_area).lazy(),
        left_on="community_area",
        right_on="AREA_NUM_1",
        how="left"
    )
    .with_columns([pl.col(col).fill_null(0) for col in POI_COLUMNS])
)

Community-area weather-station assignment:
weather_station
IGQ     4
MDW    57
ORD    16
Name: count, dtype: int64


### Function to aggregate taxi data into the spatio-temporal datasets

In [15]:
def add_taxi_data(
    skeleton: pl.LazyFrame | pl.DataFrame,
    taxi: pl.LazyFrame | pl.DataFrame,
    spatial_col: str,
    taxi_spatial_col: str,
    time_unit: str,
    timestamp_col: str = "trip_start_timestamp",
    datetime_col: str = "datetime_hour",
) -> pl.LazyFrame:
    """
    Adds hourly trip-start demand to a complete hour × spatial-unit skeleton.

    Parameters
    ----------
    skeleton:
        Complete base dataset with one row per datetime_col × spatial_col.
        Example: datetime_hour × h3_cell, datetime_hour × census_tract, etc.

    taxi:
        Taxi trip dataset with one row per trip.

    spatial_col:
        Spatial column name in the skeleton/output.
        Example: "h3_cell", "census_tract", "community_area".

    taxi_spatial_col:
        Spatial pickup column name in taxi.
        Example: "pickup_h3_cell", "pickup_census_tract", "pickup_community_area".

    timestamp_col:
        Taxi trip start timestamp column.

    datetime_col:
        Hourly timestamp column used in skeleton/output.

    demand_col:
        Output demand column name.

    Returns
    -------
    pl.LazyFrame
        Skeleton enriched with demand_col.
    """

    if isinstance(skeleton, pl.DataFrame):
        skeleton = skeleton.lazy()

    if isinstance(taxi, pl.DataFrame):
        taxi = taxi.lazy()

    taxi_demand = (
        taxi
        .filter(
            pl.col(timestamp_col).is_not_null()
            & pl.col(taxi_spatial_col).is_not_null() # TODO Census Tract contains nulls
        )
        .with_columns([
            pl.col(timestamp_col)
            .cast(pl.Datetime("us"))
            .dt.truncate(time_unit)
            .alias(datetime_col),

            pl.col(taxi_spatial_col).alias(spatial_col),
        ])
        .group_by([
            datetime_col,
            spatial_col,
        ])
        .agg(
            # aggregations:
            pl.len().alias('trip_count'),
            pl.col('trip_seconds').sum().alias('trip_seconds_sum'),
            pl.col('trip_seconds').mean().alias('trip_seconds_mean'),
            pl.col('trip_seconds').min().alias('trip_seconds_min'),
            pl.col('trip_seconds').max().alias('trip_seconds_max'),
            pl.col('trip_miles').sum().alias('trip_miles_sum'),
            pl.col('trip_miles').mean().alias('trip_miles_mean'),
            pl.col('trip_miles').min().alias('trip_miles_min'),
            pl.col('trip_miles').max().alias('trip_miles_max'),
            pl.col('fare').sum().alias('fare_sum'),
            pl.col('fare').mean().alias('fare_mean'),
            pl.col('fare').min().alias('fare_min'),
            pl.col('fare').max().alias('fare_max'),
            pl.col('tips').sum().alias('tips_sum'),
            pl.col('tips').mean().alias('tips_mean'),
            pl.col('tips').min().alias('tips_min'),
            pl.col('tips').max().alias('tips_max'),
            pl.col('tolls').sum().alias('tolls_sum'),
            pl.col('tolls').mean().alias('tolls_mean'),
            pl.col('tolls').min().alias('tolls_min'),
            pl.col('tolls').max().alias('tolls_max'),
            pl.col('extras').sum().alias('extras_sum'),
            pl.col('extras').mean().alias('extras_mean'),
            pl.col('extras').min().alias('extras_min'),
            pl.col('extras').max().alias('extras_max'),
            pl.col('trip_total').sum().alias('trip_total_sum'),
            pl.col('trip_total').mean().alias('trip_total_mean'),
            pl.col('trip_total').min().alias('trip_total_min'),
            pl.col('trip_total').max().alias('trip_total_max'),
            pl.col('payment_type').drop_nulls().mode().first().alias('most_common_payment_type')
        )    
    )

    result = (
        skeleton
        .with_columns(
            pl.col(datetime_col)
            .cast(pl.Datetime("us"))
            .dt.truncate(time_unit)
            .alias(datetime_col)
        )
        .join(
            taxi_demand,
            on=[datetime_col, spatial_col],
            how="left",
        )
        .with_columns([
            pl.col("trip_count").fill_null(0).alias("trip_count"),

            pl.col("trip_seconds_sum").fill_null(0).alias("trip_seconds_sum"),
            pl.col("trip_seconds_mean").fill_null(0).alias("trip_seconds_mean"),
            pl.col("trip_seconds_min").fill_null(0).alias("trip_seconds_min"),
            pl.col("trip_seconds_max").fill_null(0).alias("trip_seconds_max"),

            pl.col("trip_miles_sum").fill_null(0).alias("trip_miles_sum"),
            pl.col("trip_miles_mean").fill_null(0).alias("trip_miles_mean"),
            pl.col("trip_miles_min").fill_null(0).alias("trip_miles_min"),
            pl.col("trip_miles_max").fill_null(0).alias("trip_miles_max"),

            pl.col("fare_sum").fill_null(0).alias("fare_sum"),
            pl.col("fare_mean").fill_null(0).alias("fare_mean"),
            pl.col("fare_min").fill_null(0).alias("fare_min"),
            pl.col("fare_max").fill_null(0).alias("fare_max"),

            pl.col("tips_sum").fill_null(0).alias("tips_sum"),
            pl.col("tips_mean").fill_null(0).alias("tips_mean"),
            pl.col("tips_min").fill_null(0).alias("tips_min"),
            pl.col("tips_max").fill_null(0).alias("tips_max"),

            pl.col("tolls_sum").fill_null(0).alias("tolls_sum"),
            pl.col("tolls_mean").fill_null(0).alias("tolls_mean"),
            pl.col("tolls_min").fill_null(0).alias("tolls_min"),
            pl.col("tolls_max").fill_null(0).alias("tolls_max"),

            pl.col("extras_sum").fill_null(0).alias("extras_sum"),
            pl.col("extras_mean").fill_null(0).alias("extras_mean"),
            pl.col("extras_min").fill_null(0).alias("extras_min"),
            pl.col("extras_max").fill_null(0).alias("extras_max"),

            pl.col("trip_total_sum").fill_null(0).alias("trip_total_sum"),
            pl.col("trip_total_mean").fill_null(0).alias("trip_total_mean"),
            pl.col("trip_total_min").fill_null(0).alias("trip_total_min"),
            pl.col("trip_total_max").fill_null(0).alias("trip_total_max"),
        ])
        .with_columns(
            pl.col("most_common_payment_type")
            .fill_null("No trips")
            .alias("most_common_payment_type")
        )
    )

    return result

In [16]:
taxi_hexagons = {}
for resolution in H3_RESOLUTION:
    pickup_col = f"pickup_h3_cell_{resolution}"
    taxi_hexagons[resolution] = (
        scan_taxi_in_configured_period(SILVER_TAXI_HEXAGON_PATHS[resolution])
        .with_columns(
            h3_expr(
                "pickup_centroid_latitude",
                "pickup_centroid_longitude",
                resolution,
                pickup_col,
            )
        )
        .join(
            pl.scan_parquet(SILVER_HEXAGON_PATHS[resolution]).select("h3_cell"),
            left_on=pickup_col,
            right_on="h3_cell",
            how="inner",
        )
    )
taxi_census = scan_taxi_in_configured_period(SILVER_TAXI_CENSUS_PATH)
taxi_community = scan_taxi_in_configured_period(SILVER_TAXI_COMMUNITY_PATH)
taxi_community_unfiltered = scan_taxi_in_configured_period(
    SILVER_TAXI_COMMUNITY_UNFILTERED_PATH
)
generated_outputs = {}

for time_unit in GOLD_TIME_UNITS:
    print(f"\nGenerating {time_unit} Gold demand datasets...")
    temporal_features = create_temporal_features(time_unit)
    weather_features = create_weather_features(time_unit)
    (
        hexagon_outputs,
        census_output,
        community_output,
        community_unfiltered_output,
    ) = GOLD_DEMAND_PATHS[time_unit]

    generated_outputs[time_unit] = {}
    for resolution, hexagon_output in hexagon_outputs.items():
        hexagon_demand = add_taxi_data(
            skeleton=attach_nearest_station_weather(
                temporal_features, hexagon_contexts[resolution], weather_features
            ),
            taxi=taxi_hexagons[resolution],
            spatial_col="h3_cell",
            taxi_spatial_col=f"pickup_h3_cell_{resolution}",
            time_unit=time_unit,
        )
        hexagon_demand.sink_parquet(hexagon_output)
        generated_outputs[time_unit][f"hexagon_{resolution}"] = hexagon_output

    census_demand = add_taxi_data(
        skeleton=(
            attach_nearest_station_weather(
                temporal_features, census_tracts_with_pois, weather_features
            )
            .with_columns(pl.col("census_tract").cast(pl.Int64))
        ),
        taxi=taxi_census,
        spatial_col="census_tract",
        taxi_spatial_col="pickup_census_tract",
        time_unit=time_unit,
    )
    census_demand.sink_parquet(census_output)

    community_demand = add_taxi_data(
        skeleton=(
            attach_nearest_station_weather(
                temporal_features, community_area_with_pois, weather_features
            )
            .with_columns(pl.col("community_area").cast(pl.Int64))
        ),
        taxi=taxi_community,
        spatial_col="community_area",
        taxi_spatial_col="pickup_community_area",
        time_unit=time_unit,
    )
    community_demand.sink_parquet(community_output)

    community_unfiltered_demand = add_taxi_data(
        skeleton=(
            attach_nearest_station_weather(
                temporal_features, community_area_with_pois, weather_features
            )
            .with_columns(pl.col("community_area").cast(pl.Int64))
        ),
        taxi=taxi_community_unfiltered,
        spatial_col="community_area",
        taxi_spatial_col="pickup_community_area",
        time_unit=time_unit,
    )
    community_unfiltered_demand.sink_parquet(community_unfiltered_output)

    generated_outputs[time_unit].update({
        "census_tract": census_output,
        "community_area": community_output,
        "community_area_unfiltered": community_unfiltered_output,
    })
    print(f"Finished {time_unit}: {generated_outputs[time_unit]}")


Generating 1h Gold demand datasets...
Finished 1h: {'hexagon_7': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/GOLD_1H_DEMAND_HEXAGON_7.parquet'), 'hexagon_8': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/GOLD_1H_DEMAND_HEXAGON_8.parquet'), 'census_tract': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/GOLD_1H_DEMAND_CENSUS_TRACTS.parquet'), 'community_area': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet'), 'community_area_unfiltered': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/GOLD_1H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet')}

Generating 4h Gold demand datasets...
Finished 4h: {'hexagon_7': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/processed_data/GOLD_4H_DEMAND_HEXAGON_7.parquet'), 'hexagon_8': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/proce

## H3 resolution comparison datasets (spatial-resolution variation)

The main pipeline above generates every configured temporal resolution for H3 resolutions 7 and 8. The optional cells below support additional spatial-resolution experiments, currently resolution 9.

- Additional city grids are derived from the finest configured Chicago grid via `cell_to_parent` / `cell_to_children`, so the covered area is consistent.
- POI counts are re-aggregated at each resolution from the raw POI coordinates.
- Taxi pickups are re-indexed from pickup coordinates at the requested resolution.
- Outputs: `GOLD_{time}_DEMAND_HEXAGON_R{res}.parquet` in the processed directory.

Generation is gated behind `GENERATE_H3_RESOLUTION_COMPARISON` because the res-9 hourly grid is large. Enable it before running the SVM/NN spatial-resolution experiments.

In [17]:
H3_COMPARISON_RESOLUTIONS = (9,)  # resolutions 7 and 8 are produced above
GENERATE_H3_RESOLUTION_COMPARISON = False  # opt-in: large outputs, see markdown above

POI_FEATURES = ["food_drink", "landmark", "shop", "train_station"]


def make_h3_spatial_context(resolution: int) -> pl.LazyFrame:
    """Chicago H3 grid at the requested resolution with POI counts attached."""
    base_resolution = max(H3_RESOLUTION)
    base_cells = (
        pl.read_parquet(SILVER_HEXAGON_PATHS[base_resolution])
        .get_column("h3_cell").drop_nulls().unique().to_list()
    )
    if resolution in H3_RESOLUTION:
        cells_at_res = sorted(
            pl.read_parquet(SILVER_HEXAGON_PATHS[resolution])
            .get_column("h3_cell").drop_nulls().unique().to_list()
        )
    elif resolution < base_resolution:
        cells_at_res = sorted({h3.cell_to_parent(cell, resolution) for cell in base_cells})
    else:
        cells_at_res = sorted({
            child for cell in base_cells for child in h3.cell_to_children(cell, resolution)
        })

    poi_frame = pois[["lat", "lon", "poi_category"]].dropna(subset=["lat", "lon"]).copy()
    poi_frame["h3_cell"] = [
        h3.latlng_to_cell(lat, lon, resolution)
        for lat, lon in zip(poi_frame["lat"], poi_frame["lon"])
    ]
    poi_counts = (
        poi_frame.groupby(["h3_cell", "poi_category"]).size().unstack(fill_value=0).reset_index()
    )
    for col in POI_FEATURES:
        if col not in poi_counts:
            poi_counts[col] = 0

    spatial_points = pd.DataFrame({"h3_cell": cells_at_res})
    coordinates = [h3.cell_to_latlng(cell) for cell in cells_at_res]
    spatial_points["latitude"] = [latitude for latitude, _ in coordinates]
    spatial_points["longitude"] = [longitude for _, longitude in coordinates]
    station_assignment = nearest_weather_station_assignment(
        spatial_points,
        id_column="h3_cell",
    )

    return (
        pl.from_pandas(station_assignment).lazy()
        .join(pl.from_pandas(poi_counts[["h3_cell", *POI_FEATURES]]).lazy(), on="h3_cell", how="left")
        .with_columns([pl.col(col).fill_null(0) for col in POI_FEATURES])
        .with_columns(pl.lit(resolution).cast(pl.Int8).alias("h3_resolution"))
    )

In [18]:
if GENERATE_H3_RESOLUTION_COMPARISON:
    resolution_outputs = {}
    for resolution in H3_COMPARISON_RESOLUTIONS:
        spatial_context = make_h3_spatial_context(resolution)
        taxi_col = f"pickup_h3_r{resolution}"
        taxi_at_res = pl.scan_parquet(GOLD_TAXI_PATH).with_columns(
            h3_expr("pickup_centroid_latitude", "pickup_centroid_longitude", resolution, taxi_col)
        )
        for time_unit in GOLD_TIME_UNITS:
            temporal_features = create_temporal_features(time_unit)
            weather_features = create_weather_features(time_unit)
            output_path = PROCESSED_DIR / f"GOLD_{time_unit.upper()}_DEMAND_HEXAGON_R{resolution}.parquet"
            demand = add_taxi_data(
                skeleton=attach_nearest_station_weather(
                    temporal_features, spatial_context, weather_features
                ),
                taxi=taxi_at_res,
                spatial_col="h3_cell",
                taxi_spatial_col=taxi_col,
                time_unit=time_unit,
            )
            print(f"Writing {output_path.name} ...")
            demand.sink_parquet(output_path)
            resolution_outputs[(resolution, time_unit)] = output_path

    for (resolution, time_unit), output_path in resolution_outputs.items():
        result = duckdb.sql(f"""
            SELECT COUNT(*) AS rows, SUM(trip_count) AS trips
            FROM read_parquet('{output_path}')
        """).fetchone()
        print(f"r{resolution} {time_unit}: {output_path.name} -> rows={result[0]:,}, trips={result[1]:,}")
else:
    print("Set GENERATE_H3_RESOLUTION_COMPARISON = True to build optional res-9 datasets.")

Set GENERATE_H3_RESOLUTION_COMPARISON = True to build optional res-9 datasets.


In [19]:
output_trip_totals = {}

for time_unit, outputs in generated_outputs.items():
    output_trip_totals[time_unit] = {}
    for spatial_unit, output_path in outputs.items():
        result = duckdb.sql(f"""
            SELECT COUNT(*) AS rows, SUM(trip_count) AS trips
            FROM read_parquet('{output_path}')
        """).fetchone()
        output_trip_totals[time_unit][spatial_unit] = result[1]
        print(time_unit, spatial_unit, output_path.name, result)

    filtered_totals = {
        *[
            output_trip_totals[time_unit][f"hexagon_{resolution}"]
            for resolution in H3_RESOLUTION
        ],
        output_trip_totals[time_unit]["census_tract"],
        output_trip_totals[time_unit]["community_area"],
    }

1h hexagon_7 GOLD_1H_DEMAND_HEXAGON_7.parquet (54768, 130206)
1h hexagon_8 GOLD_1H_DEMAND_HEXAGON_8.parquet (327936, 130206)
1h census_tract GOLD_1H_DEMAND_CENSUS_TRACTS.parquet (295008, 130206)
1h community_area GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet (25872, 130206)
1h community_area_unfiltered GOLD_1H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet (25872, 258626)
4h hexagon_7 GOLD_4H_DEMAND_HEXAGON_7.parquet (13692, 130206)
4h hexagon_8 GOLD_4H_DEMAND_HEXAGON_8.parquet (81984, 130206)
4h census_tract GOLD_4H_DEMAND_CENSUS_TRACTS.parquet (73752, 130206)
4h community_area GOLD_4H_DEMAND_COMMUNITY_AREAS.parquet (6468, 130206)
4h community_area_unfiltered GOLD_4H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet (6468, 258626)
24h hexagon_7 GOLD_24H_DEMAND_HEXAGON_7.parquet (2282, 130206)
24h hexagon_8 GOLD_24H_DEMAND_HEXAGON_8.parquet (13664, 130206)
24h census_tract GOLD_24H_DEMAND_CENSUS_TRACTS.parquet (12292, 130206)
24h community_area GOLD_24H_DEMAND_COMMUNITY_AREAS.parquet (1078, 130206)
24h co

## Quality Check

In [20]:
con = duckdb.connect()

for time_unit, outputs in generated_outputs.items():
    for spatial_unit, output_path in outputs.items():
        columns = con.sql(f"""
            DESCRIBE SELECT * FROM read_parquet('{output_path}')
        """).df()["column_name"].tolist()
        parts = [
            f"""
            SELECT '{col}' AS column_name, COUNT(*) AS n_rows,
                    SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) AS n_nulls,
                    ROUND(100.0 * SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS null_pct
            FROM read_parquet('{output_path}')
            """
            for col in columns
        ]
        query = "\nUNION ALL\n".join(parts) + "\nORDER BY null_pct DESC"
        print(f"\nNull report for {time_unit}: {output_path.name}")
        print(con.sql(query).df().to_string(index=False))


Null report for 1h: GOLD_1H_DEMAND_HEXAGON_7.parquet


                column_name  n_rows  n_nulls  null_pct
                    weekday   54768      0.0       0.0
                  tolls_max   54768      0.0       0.0
                  month_sin   54768      0.0       0.0
                       hour   54768      0.0       0.0
weather_station_distance_km   54768      0.0       0.0
                weekday_cos   54768      0.0       0.0
                   landmark   54768      0.0       0.0
                       date   54768      0.0       0.0
                       shop   54768      0.0       0.0
                   hour_sin   54768      0.0       0.0
              h3_resolution   54768      0.0       0.0
                  month_cos   54768      0.0       0.0
                    h3_cell   54768      0.0       0.0
                   hour_cos   54768      0.0       0.0
                 food_drink   54768      0.0       0.0
                       p01m   54768      0.0       0.0
              train_station   54768      0.0       0.0
          

## Trip-count comparison

`COUNT(*)` measures the size of the complete time × spatial-unit grid. The underlying taxi population is compared with `SUM(trip_count)`.

In [21]:
trip_count_comparison_rows = []

for time_unit, outputs in generated_outputs.items():
    for spatial_unit, output_path in outputs.items():
        grid_rows, trip_count_sum, nonzero_grid_rows = con.sql(f"""
            SELECT
                COUNT(*) AS grid_rows,
                SUM(trip_count) AS trip_count_sum,
                SUM(CASE WHEN trip_count > 0 THEN 1 ELSE 0 END) AS nonzero_grid_rows
            FROM read_parquet('{output_path}')
        """).fetchone()
        trip_count_comparison_rows.append({
            "time_unit": time_unit,
            "spatial_unit": spatial_unit,
            "file": output_path.name,
            "grid_rows": grid_rows,
            "nonzero_grid_rows": nonzero_grid_rows,
            "trip_count_sum": trip_count_sum,
        })

trip_count_comparison = pd.DataFrame(trip_count_comparison_rows)
community_full_totals = (
    trip_count_comparison
    .query("spatial_unit == 'community_area_unfiltered'")
    .set_index("time_unit")["trip_count_sum"]
)
trip_count_comparison["pct_of_full_community_trips"] = (
    100
    * trip_count_comparison["trip_count_sum"]
    / trip_count_comparison["time_unit"].map(community_full_totals)
)

filtered_units = [
    *[f"hexagon_{resolution}" for resolution in H3_RESOLUTION],
    "census_tract",
    "community_area",
]
filtered_trip_totals = trip_count_comparison.query(
    "spatial_unit in @filtered_units"
)

trip_count_comparison.sort_values(["time_unit", "spatial_unit"])

,time_unit,spatial_unit,file,grid_rows,nonzero_grid_rows,trip_count_sum,pct_of_full_community_trips
2,1h,census_tract,GOLD_1H_DEMAND_CENSUS_TRACTS.parquet,295008,7328,130206,50.345286
3,1h,community_area,GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet,25872,2364,130206,50.345286
4,1h,community_area_unfiltered,GOLD_1H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet,25872,13817,258626,100.000000
0,1h,hexagon_7,GOLD_1H_DEMAND_HEXAGON_7.parquet,54768,3028,130206,50.345286
1,1h,hexagon_8,GOLD_1H_DEMAND_HEXAGON_8.parquet,327936,6091,130206,50.345286
12,24h,census_tract,GOLD_24H_DEMAND_CENSUS_TRACTS.parquet,12292,955,130206,50.345286
13,24h,community_area,GOLD_24H_DEMAND_COMMUNITY_AREAS.parquet,1078,247,130206,50.345286
14,24h,community_area_unfiltered,GOLD_24H_DEMAND_COMMUNITY_AREAS_UNFILTERED.par...,1078,1072,258626,100.000000
10,24h,hexagon_7,GOLD_24H_DEMAND_HEXAGON_7.parquet,2282,320,130206,50.345286
11,24h,hexagon_8,GOLD_24H_DEMAND_HEXAGON_8.parquet,13664,769,130206,50.345286
